<a href="https://colab.research.google.com/github/commertech-official/devops-start/blob/main/work_drive_simple_deepseek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import ipywidgets as widgets

# ============================================================
# ПАРАМЕТРЫ ДВИГАТЕЛЯ (типовой асинхронный двигатель ~1 кВт)
# ============================================================
p = 2          # число пар полюсов
R_s = 5.0      # сопротивление статора, Ом
R_r = 3.0      # сопротивление ротора, Ом
L_s = 0.3      # индуктивность статора, Гн
L_r = 0.3      # индуктивность ротора, Гн
L_m = 0.28     # взаимная индуктивность, Гн

# Приведённые параметры
sigma = 1 - (L_m**2) / (L_s * L_r)   # коэффициент рассеяния
K_r = L_m / L_r                        # коэффициент связи ротора

# Момент инерции
J = 0.01       # кг·м²

# Номинальные значения
I_sd_nom = 5.0   # номинальный ток намагничивания, А
I_sq_nom = 5.0   # номинальный ток момента, А

# ============================================================
# ФУНКЦИЯ РАСЧЁТА МОМЕНТА
# ============================================================
def calculate_torque(I_sd, I_sq):
    """
    M = (3/2) * p * (L_m^2 / L_r) * I_sd * I_sq
    """
    M = 1.5 * p * (L_m**2 / L_r) * I_sd * I_sq
    return M

# ============================================================
# СИМУЛЯЦИЯ ДИНАМИКИ (упрощённая модель)
# ============================================================
def simulate_dynamics(I_sd, I_sq, t_end=2.0, dt=0.001):
    """
    Упрощённая динамическая модель:
    - Момент зависит от I_sd и I_sq
    - Скорость меняется по закону: J * dω/dt = M - M_load
    - M_load = 0 (холостой ход) или пропорционален скорости (вентилятор)
    """
    t = np.arange(0, t_end, dt)
    n = len(t)

    omega_m = np.zeros(n)   # механическая скорость, рад/с
    M = np.zeros(n)         # момент, Н·м

    # Постоянные токи (можно сделать динамику)
    M_constant = calculate_torque(I_sd, I_sq)

    # Нагрузка: вентиляторная характеристика (M_load = k * ω²)
    k_load = 0.0001

    for i in range(1, n):
        M[i] = M_constant
        M_load = k_load * omega_m[i-1]**2
        domega = (M[i] - M_load) / J * dt
        omega_m[i] = omega_m[i-1] + domega

    # Перевод в об/мин
    rpm = omega_m * 60 / (2 * np.pi)

    return t, rpm, M

# ============================================================
# ИНТЕРАКТИВНЫЙ ГРАФИК
# ============================================================
def plot_simulation(I_sd=5.0, I_sq=5.0):
    t, rpm, M = simulate_dynamics(I_sd, I_sq)
    M_steady = calculate_torque(I_sd, I_sq)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # График скорости
    axes[0].plot(t, rpm, 'b-', linewidth=2)
    axes[0].set_xlabel('Время, с')
    axes[0].set_ylabel('Скорость, об/мин')
    axes[0].set_title(f'Скорость вращения\nId = {I_sd:.1f} А, Iq = {I_sq:.1f} А')
    axes[0].grid(True, alpha=0.3)
    axes[0].axhline(y=rpm[-1], color='r', linestyle='--',
                    label=f'Установившаяся: {rpm[-1]:.0f} об/мин')
    axes[0].legend()

    # График момента
    axes[1].plot(t, M, 'g-', linewidth=2)
    axes[1].set_xlabel('Время, с')
    axes[1].set_ylabel('Момент, Н·м')
    axes[1].set_title(f'Электромагнитный момент\nM = {M_steady:.3f} Н·м')
    axes[1].grid(True, alpha=0.3)
    axes[1].axhline(y=M_steady, color='r', linestyle='--',
                    label=f'M = {M_steady:.3f} Н·м')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

# Запуск интерактивного виджета
interact(plot_simulation,
         I_sd=FloatSlider(min=0, max=10, step=0.5, value=5.0,
                          description='Id (А):'),
         I_sq=FloatSlider(min=-10, max=10, step=0.5, value=5.0,
                          description='Iq (А):'))

interactive(children=(FloatSlider(value=5.0, description='Id (А):', max=10.0, step=0.5), FloatSlider(value=5.0…

<function __main__.plot_simulation(I_sd=5.0, I_sq=5.0)>